第06回講義
========

時系列データ
---------

⾝の回りの様々なデータ
- 気温，気圧
- 株価，為替レート
- GDP, 消費者物価指数
- ⾎圧，脈拍，脳波
- 地震の波動
などは時間とともに変動しています。

このように時間とともに変動している現象の記録が**時系列データ**です。

### トレンド

時系列データの⻑期的変動の傾向をトレンドと呼びます。
- トレンドは経済動向の分析などに使われます。
- トレンドの推定⽅法には

  - 移動平均
  - 季節調整

  などがあります。⽬的や推定⽅法によって、トレンドの滑らかさは変わります。

移動平均線
--------

ランダムな要素を含み、短期的に細かい変動を含むデータから長期的な変動を分析・予測するためには、短期的な変動をある程度平均化・平滑化する必要があります。その一つとして移動平均という方法を考えてみましょう。

移動平均とは、時系列データにおいて、ある一定区間ごとの平均値を区間をずらしながら求めたものです。
移動平均にはその計算方法によって様々な種類がありますが、単純移動平均(Simple Moving Average, SMA)とは求めたい時点から直近の$n$個の(過去)データの平均値です。
$$
\bar{y}_i = \frac{y_i + y_{i-1} + y_{i-2} +\cdots + y_{i-n+1}}{n}
$$
したがって、単純移動平均の計算は初めの時点からすべて可能となるわけではなく、最低でも過去の$n$個のデータが存在する時点からということになります。($y_1,y_2,y_3,\ldots$というデータであれば、$\bar{y}_n$という単純移動平均から計算可能となります。)


<mark>練習1</mark> `tokyo-temp.csv`のデータから過去5年分の単純移動平均を求め、その結果を元の気温データと重ねてプロットするプログラムを作成しなさい。(単純移動平均のプロットは1880年から2024年までとなることに注意します。) 

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import csv

f = open('tokyo-temp.csv', 'r')

data = csv.reader(f)
header = next(f)

# 元のデータの表示
year = []
temp = []

for line in data:
    year.append( int(line[0]) )
    temp.append( float(line[1]) )

f.close()

plt.plot(year, temp)

# 単純移動平均線の表示
year2 = []
sma = []

for i in range(4, len(year)):
    year2.append( year[i] )
    ave = ( temp[i] + temp[i-1] + temp[i-2] 
           + temp[i-3] + temp[i-4] ) / 5
    sma.append(ave)

plt.plot(year2, sma)

<mark>練習 2</mark> このプログラムの後半部分を、`for`文を使ってもう少し汎用的(`n`年の単純移動平均)に書き直しなさい。

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import csv

f = open('tokyo-temp.csv', 'r')

data = csv.reader(f)
header = next(f)

# 元のデータの表示
year = []
temp = []

for line in data:
    year.append( int(line[0]) )
    temp.append( float(line[1]) )

f.close()

plt.plot(year, temp)

# 過去n年に対する単純移動平均線の表示
n = 10
year2 = []
sma = []

for i in range(n-1, len(year)):
    year2.append( year[i] )
    ave = 0
    for j in range(n):
        ave += temp[i-j] / n
    sma.append(ave)

plt.plot(year2, sma)

### TOPIX(東証株価指数)のプロット

`topix.csv`は2004年4月1日から2025年1月31日までのTOPX(東証株価指数)の始値、高値、安値、終値のデータです。

<mark>練習3</mark> `topix.csv`の終値をプロットしなさい。(横軸を日付データとして扱う。)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import csv
from datetime import datetime

f = open('topix.csv', 'r')

data = csv.reader(f)
header = next(f)

# 元のデータの表示
day = []
topix = []

for line in data:
    # datetime.strptime()で文字列をdatetime型に変換
    day.append( datetime.strptime(line[0], '%Y/%m/%d') )
    topix.append( float(line[4]) )

f.close()

plt.plot(day, topix)

時系列の周期
---------

<mark>練習 4</mark> `sunspot.csv`は各年の太陽黒点数についてのデータです。このデータを横軸を年、縦軸を太陽黒点数としてプロットしなさい。

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import csv

f = open('sunspot.csv', 'r')

data = csv.reader(f)
header = next(f)

# データの表示
year = []
sunspot = []

for line in data:
    year.append( int(line[0]) )
    sunspot.append( float(line[1]) )

f.close()

plt.plot(year, sunspot)

プロットされたものを見ると太陽黒点数は一定の周期で増減を繰り返していることがわかります。

時系列が⼀定の間隔で同じような変動を繰り返す成分を持つとき、周期的変動と呼びます。

$x_{n-p} \sim x_n$: 周期的変動  
$p$: 周期

経済指標などの時系列データではでは1年周期の変動は季節変動と呼ばれます。

現実の時系列では完全な周期変動は少なく、周期的変動パターンが徐々に変化することがあります。完全な周期関数の推定は簡単ですが、少しずつ変化する周期的成分の推定は難しい問題となります。

階差
---

周期変動が存在するとき、すべての$i$に対して、
$$\Delta_p x_i \equiv x_i - x_{i-p} \sim 0$$
が成り立つはずです。(あるいはその絶対値の平均がゼロに近づく。)

$\Delta_p x_i$のことを階差(差分)といいます。(正確には$p=1$のときを階差といい、季節変動に合わせて$p$を選んだものを季節階差といいます。)

<mark>練習 5</mark> 太陽黒点数についてのデータ`sunspot.csv`を用いて、与えられた$p$に対して階差を計算し、その階差をプロットしなさい。

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import csv

f = open('sunspot.csv', 'r')

data = csv.reader(f)
header = next(f)

# データの表示
year = []
sunspot = []

for line in data:
    year.append( int(line[0]) )
    sunspot.append( float(line[1]) )

f.close()

# pの値を設定
p = 5

# 差分を計算したものを格納するリスト
diff = []
year2 = []

for i in range(p, len(sunspot)):
  diff.append( sunspot[i] - sunspot[i-p] )
  year2.append( year[i] )

plt.plot(year2, diff)

<mark>練習 6</mark> $p$を1から25まで変化させて、階差が変化する様子を調べなさい。(階差分散の値を求め、プロットしてみる。)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import csv

f = open('sunspot.csv', 'r')

data = csv.reader(f)
header = next(f)

# データの表示
year = []
sunspot = []

for line in data:
    year.append( int(line[0]) )
    sunspot.append( float(line[1]) )

f.close()

variance = []

for p in range(1, 26):

  diff = []
  year2 = []

  for i in range(p, len(sunspot)):
    diff.append( sunspot[i] - sunspot[i-p] )
    year2.append( year[i] )

  plt.plot(year2, diff, label='p='+str(p))
  variance.append( np.var(diff) )
  
plt.legend()
plt.show()

plt.plot(range(1, 26), variance)

自己相関
------

時系列データ$x_i$と$p$だけ離れた時刻のデータ$x_{i-p}$の相関係数
$$
R(p) = r(x_{i}, x_{i-p})
$$
を$p$の関数と見たものを**自己相関関数**といいます。

<mark>練習 7</mark> `sunspot.csv`に対して、横軸を$x_i$のデータ、縦軸を$x_{i-p}$のデータとして、$p$の値を変化させながら、散布図をプロットしなさい。(合わせて相関係数も出力する。)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import csv

f = open('sunspot.csv', 'r')

data = csv.reader(f)
header = next(f)

# データの収納
sunspot = []

for line in data:
    sunspot.append( float(line[1]) )

f.close()

p = 12

original = []
shifted = []

for i in range(p, len(sunspot)):
  original.append( sunspot[i] )
  shifted.append( sunspot[i-p] )

plt.scatter(original, shifted, label='p='+str(p))
plt.xlabel('original')
plt.ylabel('shifted')
plt.legend()

print( np.corrcoef(original, shifted) )

<mark>練習 8</mark> `sunspot.csv`に対して、横軸を$p$ ($p=1,2,3,\cdots,50)$として自己相関関数をプロットしなさい。

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import csv

f = open('sunspot.csv', 'r')

data = csv.reader(f)
header = next(f)

# データの収納
sunspot = []

for line in data:
    sunspot.append( float(line[1]) )

f.close()

corr = []

for p in range(1, 51):

  original = []
  shifted = []

  for i in range(p, len(sunspot)):
    original.append( sunspot[i] )
    shifted.append( sunspot[i-p] )

  corr.append( np.corrcoef(original, shifted)[0][1] )

plt.plot(range(1, 51), corr)